# Starter — a check that fails loudly

*Replace this notebook. It exists to show the house rules working, not to be kept.*

The rules this demonstrates are this folder's:

> Schema changes are versioned, ordered and idempotent — re-running a migration
> must not change row counts.
>
> Prefer a check that fails loudly over a report nobody reads.


In [ ]:
import os

import psycopg

# Read-only against a tier you share with everyone else on this machine.
DSN = ('host={PGHOST} port={PGPORT} user={PGUSER} '
       'password={PGPASSWORD} dbname={PGDATABASE}').format(**os.environ)
conn = psycopg.connect(DSN)
conn.read_only = True


## Is the migration ledger honest?

`schema_migrations` is the claim. The tables are the fact. A ledger that has
recorded migrations nobody can see is worse than no ledger.


In [ ]:
applied = conn.execute('select * from schema_migrations order by 1').fetchall()
for row in applied:
    print(' ', row)

assert applied, 'nothing recorded in schema_migrations; the seed did not run'
versions = [r[0] for r in applied]
assert len(versions) == len(set(versions)), f'a migration is recorded twice: {versions}'


## Is it idempotent, or just untested?

Counting twice does not prove idempotency on its own — but a count that moves
while nothing is writing proves the opposite, and costs one query to find out.


In [ ]:
TABLES = ['knowledge_asset', 'critical_data_element', 'person', 'sop_version']
before = {t: conn.execute(f'select count(*) from "{t}"').fetchone()[0] for t in TABLES}
after = {t: conn.execute(f'select count(*) from "{t}"').fetchone()[0] for t in TABLES}
assert before == after, f'row counts moved with nothing writing: {before} -> {after}'
before


## A data-quality check with somewhere to go

Every critical data element is supposed to have a named owner. An unowned CDE
is a governance gap, not a null — so this fails rather than reporting.


In [ ]:
orphans = conn.execute('''
    select cde.element_name
      from critical_data_element cde
      left join person p on p.id = cde.owner_id
     where p.id is null
''').fetchall()
assert not orphans, f'critical data elements with no owner: {[o[0] for o in orphans]}'
print('every CDE has an owner')


## The other two services

Redis and MinIO are on the same tier. The round-trip below writes to a key
namespaced to this notebook and removes it, so nothing is left behind.


In [ ]:
import uuid

import redis

r = redis.Redis(host=os.environ['ACME_REDIS_HOST'], port=int(os.environ['ACME_REDIS_PORT']))
key = f'lab:starter:{uuid.uuid4()}'
r.set(key, 'round trip', ex=60)
assert r.get(key) == b'round trip', 'redis accepted a write it cannot read back'
r.delete(key)
assert r.get(key) is None, 'the probe key outlived the notebook'
print('redis round-trip clean, nothing left behind')


In [ ]:
conn.close()
